## Extração de Features — Key Points

Calcula ratios de desempenho nos pontos decisivos: break points enfrentados no saque (KPS) e oportunidades de quebra no retorno (KPR).
Ao final, salva `features_keypoints.csv` com as médias acumuladas pré-partida de cada jogador.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

BASE = '../../new-dataset/tennis_MatchChartingProject-master'

kps = pd.read_csv(f'{BASE}/charting-m-stats-KeyPointsServe.csv')
kpr = pd.read_csv(f'{BASE}/charting-m-stats-KeyPointsReturn.csv')

print('kps:', kps.shape)
print('kpr:', kpr.shape)
kps.head(4)

kps: (60464, 12)
kpr: (60464, 8)


,match_id,player,row,pts,pts_won,first_in,aces,svc_winners,rally_winners,rally_forced,unforced,dfs
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,BP,11,8,6,1,1,2,2,1,1
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Michael Zheng,BP,3,3,2,0,1,0,1,0,0
2,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,GP,13,8,10,1,1,2,1,2,1
3,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Michael Zheng,GP,14,10,10,0,1,6,2,2,0


#### Calculando ratios e adicionando data

In [2]:
kps['bp_clutch_save_pct'] = kps['pts_won'] / kps['pts'].replace(0, np.nan)
kps['bp_first_in_pct']    = kps['first_in'] / kps['pts'].replace(0, np.nan)

kpr['bpo_conv_pct'] = kpr['pts_won']  / kpr['pts'].replace(0, np.nan)
kpr['bpo_ue_pct']   = kpr['unforced'] / kpr['pts'].replace(0, np.nan)

kps['Date'] = pd.to_datetime(
    kps['match_id'].str.split('-').str[0], format='%Y%m%d', errors='coerce'
)
kpr['Date'] = pd.to_datetime(
    kpr['match_id'].str.split('-').str[0], format='%Y%m%d', errors='coerce'
)

print(kps.shape, kpr.shape)

(60464, 15) (60464, 11)


#### Agregando por partida e jogador (somando pontos antes de calcular as médias)

In [3]:
kps_agg = (kps.groupby(['match_id', 'player', 'Date'], as_index=False)
              [['pts', 'pts_won', 'first_in']]
              .sum())

kpr_agg = (kpr.groupby(['match_id', 'player', 'Date'], as_index=False)
              [['pts', 'pts_won', 'unforced']]
              .sum())

kps_agg['bp_clutch_save_pct'] = kps_agg['pts_won']  / kps_agg['pts'].replace(0, np.nan)
kps_agg['bp_first_in_pct']    = kps_agg['first_in'] / kps_agg['pts'].replace(0, np.nan)

kpr_agg['bpo_conv_pct'] = kpr_agg['pts_won']  / kpr_agg['pts'].replace(0, np.nan)
kpr_agg['bpo_ue_pct']   = kpr_agg['unforced'] / kpr_agg['pts'].replace(0, np.nan)

print(kps_agg.shape, kpr_agg.shape)
kps_agg.head(3)

(15090, 8) (15090, 8)


,match_id,player,Date,pts,pts_won,first_in,bp_clutch_save_pct,bp_first_in_pct
0,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Luis Ayala,1960-05-29,66,28,40,0.424242,0.606061
1,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Nicola Pietrangeli,1960-05-29,74,36,50,0.486486,0.675676
2,19600704-M-Wimbledon-F-Rod_Laver-Neale_Fraser,Neale Fraser,1960-07-04,78,54,58,0.692308,0.743590


#### Calculando médias acumuladas pré-partida

In [4]:
kps_agg = kps_agg.sort_values(['player', 'Date'])
kpr_agg = kpr_agg.sort_values(['player', 'Date'])

for col in ['bp_clutch_save_pct', 'bp_first_in_pct']:
    kps_agg[f'avg_{col}'] = (
        kps_agg.groupby('player')[col].transform(lambda x: x.expanding().mean().shift(1))
    )

for col in ['bpo_conv_pct', 'bpo_ue_pct']:
    kpr_agg[f'avg_{col}'] = (
        kpr_agg.groupby('player')[col].transform(lambda x: x.expanding().mean().shift(1))
    )

print(kps_agg.shape, kpr_agg.shape)

(15090, 10) (15090, 10)


#### Juntando KPS e KPR em um único DataFrame

In [5]:
kps_avg = kps_agg[['match_id', 'player', 'Date', 'avg_bp_clutch_save_pct', 'avg_bp_first_in_pct']]
kpr_avg = kpr_agg[['match_id', 'player', 'Date', 'avg_bpo_conv_pct', 'avg_bpo_ue_pct']]

features_keypoints = kps_avg.merge(kpr_avg, on=['match_id', 'player', 'Date'], how='outer')

print(features_keypoints.shape)
features_keypoints.head(4)

(15090, 7)


,match_id,player,Date,avg_bp_clutch_save_pct,avg_bp_first_in_pct,avg_bpo_conv_pct,avg_bpo_ue_pct
0,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Luis Ayala,1960-05-29,NaN,NaN,NaN,NaN
1,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Nicola Pietrangeli,1960-05-29,NaN,NaN,NaN,NaN
2,19600704-M-Wimbledon-F-Rod_Laver-Neale_Fraser,Neale Fraser,1960-07-04,NaN,NaN,NaN,NaN
3,19600704-M-Wimbledon-F-Rod_Laver-Neale_Fraser,Rod Laver,1960-07-04,NaN,NaN,NaN,NaN


#### Salvando

In [6]:
features_keypoints.to_csv(f'{BASE}/features_keypoints.csv', index=False)
print('Salvo:', f'{BASE}/features_keypoints.csv')
print('Shape:', features_keypoints.shape)

Salvo: ../../new-dataset/tennis_MatchChartingProject-master/features_keypoints.csv
Shape: (15090, 7)
